# 01 - SQL Server → MinIO (Landing Zone)

Lê todas as tabelas do `LojaVirtualDB` e grava no bucket `landing-zone` do MinIO no formato CSV.

```
SQL Server (LojaVirtualDB)  →  MinIO / landing-zone / <tabela> / <tabela>.csv
```

> Execute o notebook `00_setup_sqlserver.ipynb` antes deste.


In [ ]:
import pyodbc
import pandas as pd
import boto3
from io import StringIO
from botocore.exceptions import ClientError

# SQL Server
DRIVER   = "{ODBC Driver 18 for SQL Server}"
SERVER   = "localhost,1433"
DATABASE = "LojaVirtualDB"
USER     = "sa"
PASSWORD = "SqlServer@2026!"

# MinIO
MINIO_ENDPOINT = "http://localhost:9020"
MINIO_ACCESS   = "minioadmin"
MINIO_SECRET   = "minioadmin"
LANDING_BUCKET = "landing-zone"

In [ ]:
# Lê as tabelas do SQL Server para DataFrames
conn = pyodbc.connect(
    f"DRIVER={DRIVER};SERVER={SERVER};DATABASE={DATABASE};UID={USER};PWD={PASSWORD};TrustServerCertificate=yes;"
)

tabelas = ["clientes", "produtos", "pedidos"]
dados = {t: pd.read_sql(f"SELECT * FROM {t}", conn) for t in tabelas}
conn.close()

for nome, df in dados.items():
    print(f"{nome}: {len(df)} registros | colunas: {list(df.columns)}")

In [ ]:
# Conecta ao MinIO e garante que o bucket landing-zone existe
s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS,
    aws_secret_access_key=MINIO_SECRET,
)

try:
    s3.create_bucket(Bucket=LANDING_BUCKET)
    print(f"Bucket '{LANDING_BUCKET}' criado.")
except ClientError as e:
    code = e.response["Error"]["Code"]
    if code in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print(f"Bucket '{LANDING_BUCKET}' já existia.")
    else:
        raise

In [ ]:
# Converte os DataFrames para CSV e envia ao MinIO
for nome, df in dados.items():
    buffer = StringIO()
    df.to_csv(buffer, index=False)
    key = f"{nome}/{nome}.csv"
    s3.put_object(
        Bucket=LANDING_BUCKET,
        Key=key,
        Body=buffer.getvalue().encode("utf-8"),
    )
    print(f"Enviado → s3://{LANDING_BUCKET}/{key}  ({len(df)} linhas)")

In [ ]:
# Lista os objetos no bucket para confirmar o envio
resp = s3.list_objects_v2(Bucket=LANDING_BUCKET)
print(f"Conteúdo do bucket '{LANDING_BUCKET}':")
for obj in resp.get("Contents", []):
    print(f"  {obj['Key']}  ({obj['Size']} bytes)")